# Library

In [8]:
import os

import psycopg

# Database Logging

Elasticsearch stores the knowledge; postgres stores the history. Every
question the app answers becomes one row in conversations: question,
answer, the model that produced it, tokens, cost, response time and a
timezone aware timestamp. Every thumbs up or down becomes a row in
feedback pointing at its conversation. The llm judge will later write
its verdicts next to them, and grafana reads all of it to draw the
dashboards.

The module follows the package pattern: one class, ConversationLog,
born with its connection settings, the same dependency injection shape
as the searcher and the indexer (and the same idea behind langchain's
own postgres chat histories). Nothing inside it reads the environment;
whoever builds it decides where the diary lives. Each method opens one
short-lived connection, the safe pattern under streamlit reruns.

create_tables is idempotent because make init-db runs it on every
fresh clone, and recreate=True is the explicit reset for schema
changes, mirroring the elasticsearch indexer: destructive, so it only
runs when asked by name (make reset-db). A source column marks which
channel produced each row, so future entry points (a telegram bot, an
editor plugin) can share the same table.

In [9]:
class ConversationLog:
    """The diary over postgres: every answered question is one row in
    conversations, every thumbs up or down one row in feedback, and
    grafana reads both to draw the dashboards.

    Same shape as the rest of the package: dependencies arrive at
    birth, nothing in here reads the environment. Each method opens
    one short-lived connection, so no connection object lingers
    between streamlit reruns."""

    def __init__(self, host, user, password, dbname):
        self.host = host
        self.user = user
        self.password = password
        self.dbname = dbname

    def _connect(self):
        """One connection per operation, used as a context manager so
        commit and close are automatic."""
        return psycopg.connect(host=self.host, user=self.user,
                               password=self.password, dbname=self.dbname)

    def create_tables(self, recreate=False):
        """IF NOT EXISTS keeps this safe to run on every startup: a
        fresh clone gets the schema, an existing database is left
        untouched.

        recreate=True is the explicit reset, same pattern as the
        elasticsearch indexer: drops both tables (feedback first, it
        references conversations) and rebuilds them. It erases the
        whole conversation history, so it only runs when asked by
        name (make reset-db)."""
        with self._connect() as conn:
            if recreate:
                conn.execute("DROP TABLE IF EXISTS feedback;")
                conn.execute("DROP TABLE IF EXISTS conversations;")
            conn.execute("""
                CREATE TABLE IF NOT EXISTS conversations (
                    id                SERIAL PRIMARY KEY,
                    question          TEXT NOT NULL,
                    answer            TEXT NOT NULL,
                    model             TEXT NOT NULL,
                    prompt_tokens     INTEGER,
                    completion_tokens INTEGER,
                    cost              NUMERIC(10, 6),
                    response_time     REAL,
                    source            TEXT NOT NULL DEFAULT 'streamlit',
                    created_at        TIMESTAMPTZ NOT NULL DEFAULT now()
                );
            """)
            conn.execute("""
                CREATE TABLE IF NOT EXISTS feedback (
                    id              SERIAL PRIMARY KEY,
                    conversation_id INTEGER NOT NULL REFERENCES conversations(id),
                    thumbs          INTEGER NOT NULL,
                    created_at      TIMESTAMPTZ NOT NULL DEFAULT now()
                );
            """)

    def save_conversation(self, question, answer, model, prompt_tokens,
                          completion_tokens, cost, response_time,
                          source="streamlit"):
        """One answered question becomes one row. Returns the generated
        id, which is what feedback and judge verdicts point at later."""
        with self._connect() as conn:
            row = conn.execute(
                """
                INSERT INTO conversations
                    (question, answer, model, prompt_tokens,
                     completion_tokens, cost, response_time, source)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                RETURNING id
                """,
                (question, answer, model, prompt_tokens,
                 completion_tokens, cost, response_time, source),
            ).fetchone()
        return row[0]

    def save_feedback(self, conversation_id, thumbs):
        """A thumbs up (1) or down (-1) pointing at its conversation."""
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO feedback (conversation_id, thumbs) VALUES (%s, %s)",
                (conversation_id, thumbs),
            )

In [10]:
log = ConversationLog(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    user=os.getenv("POSTGRES_USER", "user"),
    password=os.getenv("POSTGRES_PASSWORD", "pswd"),
    dbname=os.getenv("APP_POSTGRES_DB", "obsidian_assistant"),
)

with log._connect() as conn:
    version = conn.execute("SELECT version()").fetchone()[0]
print(version[:60])

PostgreSQL 18.4 (Debian 18.4-1.pgdg13+1) on x86_64-pc-linux-


In [11]:
log.create_tables(recreate=True)   # once: replaces any old-schema tables
log.create_tables()
log.create_tables()                # twice more: idempotence is the contract

with log._connect() as conn:
    tables = conn.execute(
        "SELECT tablename FROM pg_tables WHERE schemaname = 'public'"
    ).fetchall()
    columns = conn.execute(
        """SELECT column_name FROM information_schema.columns
           WHERE table_name = 'conversations' ORDER BY ordinal_position"""
    ).fetchall()
print("tables: ", [t[0] for t in tables])
print("columns:", [c[0] for c in columns])

tables:  ['conversations', 'feedback']
columns: ['id', 'question', 'answer', 'model', 'prompt_tokens', 'completion_tokens', 'cost', 'response_time', 'source', 'created_at']


In [12]:
# a fake conversation exercises the whole write path end to end
fake_id = log.save_conversation(
    question="fake question, delete me",
    answer="fake answer",
    model="test-model",
    prompt_tokens=100,
    completion_tokens=20,
    cost=0.000123,
    response_time=1.5,
)
log.save_feedback(fake_id, 1)
print("conversation id:", fake_id)

conversation id: 1


In [13]:
with log._connect() as conn:
    convs = conn.execute(
        """SELECT id, question, model, cost, response_time, source, created_at
           FROM conversations ORDER BY id DESC LIMIT 3"""
    ).fetchall()
    fb = conn.execute(
        "SELECT id, conversation_id, thumbs, created_at FROM feedback ORDER BY id DESC LIMIT 3"
    ).fetchall()

print("conversations:")
for row in convs:
    print("  ", row)
print("feedback:")
for row in fb:
    print("  ", row)

conversations:
   (1, 'fake question, delete me', 'test-model', Decimal('0.000123'), 1.5, 'streamlit', datetime.datetime(2026, 8, 7, 19, 39, 42, 860077, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')))
feedback:
   (1, 1, 1, datetime.datetime(2026, 8, 7, 19, 39, 42, 885532, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')))


In [14]:
# the diary must not be born dirty: remove the fake rows (feedback
# first, it references the conversation) and confirm both are empty
with log._connect() as conn:
    conn.execute("DELETE FROM feedback WHERE conversation_id = %s", (fake_id,))
    conn.execute("DELETE FROM conversations WHERE id = %s", (fake_id,))
    counts = conn.execute(
        "SELECT (SELECT count(*) FROM conversations), (SELECT count(*) FROM feedback)"
    ).fetchone()
print(f"conversations: {counts[0]} rows | feedback: {counts[1]} rows")

conversations: 0 rows | feedback: 0 rows
